# 1. Data acquisition and quality control

Loads the observational record for the three Washington-coast candidate sites
and establishes what is actually usable.

The buoys are NDBC 46041 (Cape Elizabeth), 46087 (Neah Bay) and 46029 (Columbia
River Bar). They are real moored buoys, so this is measurement rather than model
output — the reference everything else is checked against.

**What this notebook is not.** It does no analysis. Loading, cleaning and
coverage reporting live in `src/data/ndbc.py` and are covered by tests; this
notebook calls that code and shows the result. Analysis logic that lives only in
a notebook cannot be tested, and this project already had one bug that survived
because of exactly that.

In [ ]:
# Run from anywhere in the repo: put the project root on the path.
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (11, 5)
print(f"project root: {root}")

## Load

`load_station` downloads and caches the raw NDBC archive files, converts the
per-column missing-value sentinels to NaN, drops physically impossible values,
and resamples onto a regular hourly index.

The sentinel handling matters more than it sounds. NDBC encodes "no data" as
all-nines padded to each column's width — `99.00` for wave height, `999` for
direction, `9999.0` for pressure. Left unconverted, a `99.00` is a ninety-nine
metre wave that silently dominates every mean and threshold downstream.

In [ ]:
from src.config import NDBC_STATIONS, PILOT_STATIONS
from src.data.ndbc import coverage_report, load_station

YEARS = list(range(2015, 2024))

records = {}
for station_id in PILOT_STATIONS:
    info = NDBC_STATIONS[station_id]
    print(f"{station_id}  {info.name}  ({info.latitude:.3f} N, {info.longitude:.3f} E)")
    records[station_id] = load_station(station_id, YEARS)

{k: len(v) for k, v in records.items()}

## Coverage

Buoys go off station. Before trusting any statistic, check how much of the
record actually exists — and where the gaps fall, since a gap concentrated in
one season biases everything computed from the record.

In [ ]:
for station_id, df in records.items():
    print(f"\n=== {station_id} {NDBC_STATIONS[station_id].name} ===")
    print(coverage_report(df).to_string())

In [ ]:
# Where are the gaps? Monthly data availability per station.
fig, axes = plt.subplots(len(records), 1, figsize=(12, 2.2 * len(records)), sharex=True)
for ax, (station_id, df) in zip(np.atleast_1d(axes), records.items()):
    monthly = df["WVHT"].notna().resample("MS").mean()
    ax.fill_between(monthly.index, 0, monthly.values, step="mid", alpha=0.7)
    ax.set_ylim(0, 1)
    ax.set_ylabel("present")
    ax.set_title(f"{station_id} {NDBC_STATIONS[station_id].name}", loc="left", fontsize=10)
plt.suptitle("Monthly data availability, WVHT", y=1.0)
plt.tight_layout()
plt.show()

Note any station with a multi-month block at zero. 46087 has a long outage
around 2020–2022; statistics for that station rest on a shorter effective
record than the others and deserve correspondingly less weight.

## The observed climate

The seasonal cycle here is the single most important feature of this coast for
an energy project: winter carries several times the power of summer, and it is
also when the sea is least accessible. Those two facts are in tension and are
what notebook 3 prices.

In [ ]:
fig, ax = plt.subplots()
for station_id, df in records.items():
    monthly = df["WVHT"].groupby(df.index.month).mean()
    ax.plot(monthly.index, monthly.values, marker="o", label=f"{station_id} {NDBC_STATIONS[station_id].name}")
ax.set_xlabel("month")
ax.set_ylabel("mean $H_s$ (m)")
ax.set_title("Seasonal cycle in significant wave height")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
summary = pd.DataFrame({
    station_id: {
        "n_hours": int(df["WVHT"].notna().sum()),
        "mean_Hs_m": df["WVHT"].mean(),
        "median_Hs_m": df["WVHT"].median(),
        "p90_Hs_m": df["WVHT"].quantile(0.90),
        "max_Hs_m": df["WVHT"].max(),
        "winter_mean_m": df.loc[df.index.month.isin([12, 1, 2]), "WVHT"].mean(),
        "summer_mean_m": df.loc[df.index.month.isin([6, 7, 8]), "WVHT"].mean(),
    }
    for station_id, df in records.items()
}).T
summary["winter_summer_ratio"] = summary["winter_mean_m"] / summary["summer_mean_m"]
summary.round(2)

## Hand-off

The cleaned records feed notebook 2, which converts them to wave power, and
notebook 3, which prices access. Nothing is written to disk here — the loader
caches the raw downloads, so re-running is cheap and the cleaning is applied
identically wherever it is called.